In [ ]:
# %pip install -r ../requirements.txt
# # !! be aware !!
# # FastPFOR compression library python binding
# # only compiles with gcc -> on windows, requires mingw compiler
# #   as per detail:
# #   https://github.com/fast-pack/FastPFOR?tab=readme-ov-file#software-requirements
# # testing so far is only done on linux
# # if there are issues on windows...
# #   it is planned to provide a
# #   precompiled dll interface to FastPFOR

In [ ]:
from io import BytesIO
import sys
sys.path.append('../../')
from mdfc import (
    MDFCompressor, MDFDecompressor
)

import asammdf, numpy as np


In [ ]:
mdf_uc_path =  '../../sample_data/Automotive-ResearchDataSet-VIF_AEGIS/mdf_uncompressed.mf4'
mdf_df1_path = '../../sample_data/Automotive-ResearchDataSet-VIF_AEGIS/mdf_deflate_1.mf4'
mdf_df2_path = '../../sample_data/Automotive-ResearchDataSet-VIF_AEGIS/mdf_deflate_2.mf4'

In [ ]:
# uncompressed MDF file
MDF_FIL = BytesIO(
    open(mdf_uc_path, 'rb').read()
)
MDF_FIL.seek(0); pass

In [ ]:
# size of uncompressed MDF file in MB
uncompressed_mdf_total_size = MDF_FIL.__sizeof__()
print(
    f'{uncompressed_mdf_total_size/1000/1000:.2f} '
    'MB Uncompressed MDF File'
)

In [ ]:
# comparison against using deflate, 
# (using asammdf parameter compression=2)
#   which i think is like:
#   "column-oriented 4MB-block compression"
DEFLATE_MDF_FIL = BytesIO(
    open(mdf_df2_path, 'rb').read()
)
DEFLATE_MDF_FIL.seek(0); pass

In [ ]:
# size of deflated MDF file in MB
deflate_mdf_total_size = DEFLATE_MDF_FIL.__sizeof__()
print(
    f'{deflate_mdf_total_size/1000/1000:.2f} '
    'MB Deflate2 MDF File'
)

In [ ]:
# ratio of deflate vs uncompressed
print(
    f'{uncompressed_mdf_total_size/deflate_mdf_total_size:.3f} '
    'CR using Deflate (MDF Standard)'
)

In [ ]:
# configurable parameters for mdfc compression,
# which is just for lossy float compression
# for lossless fp compression, set:
#   tolerance, significands, minimum_tolerance
#   = -1  (the default values)
#   TODO allow some false-y value also, or None
#        but presently, it would raise value error :(
# in this example we can use these lossy params:
lossy_fp_params = dict(
    # tolerance=0.01,  # lc framework
    tolerance_rel=0.01,  # lc framework

    
    # significands = 20,  # big number just falls back to min tolerance
    # ^ meaning: 
    #   n additional digits
    #   after the significance
    #   of the smallest value
    #       uniquely for each channel
    #   eg: 
    #       if significands = 3,
    #       if channel A min_value == 1e-5,
    #       then channel A tolerance =  1e-8
    # minimum_tolerance = 1e-4,
    # ^ meaning:
    #   minimum tolerance for all channels
    
)
lossy_time_params = dict(
    # highest precision -> 1 ns
    # real-world data might be OK at 10us
    time_resolution = '1ms'
)

In [ ]:
# test params
DO_TEST_COMPRESSION   = True
DO_TEST_DECOMPRESSION = True

In [ ]:
# %%timeit
# test compression
if DO_TEST_COMPRESSION:
    MDFC_FIL = BytesIO()
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass
    with (
        asammdf.MDF(MDF_FIL) as mdf_fil,
        MDFCompressor(MDFC_FIL, close_file_on_exit=False) as mdfc_fil
    ):
        mdfc_fil.compress_all_groups(
            mdf_fil,
            on_error='raise',
            **lossy_fp_params,
            **lossy_time_params,
        )
        # presently, must call finish function,
        #   TODO it should be done on a (successful?) __exit__
        mdfc_fil.finish()
        from copy import deepcopy
        md = deepcopy(mdfc_fil.comp_metadata)
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass

In [ ]:
# in this case, 
# 13 channels
# 5.5 seconds without GA search
# 30 seconds with 4 component GA search each one
#   maybe that can be cut down with reduced sampling
# so for 2k channels... 
# either 0.2h, or 1.3h if each fp is searched
# sounds like we need to reduce the search space :)
#   maybe 4component pipeline is too much
#   maybe the first two should be bitshuffle or diff
#   maybe with lossy compression, the search space can be less?
# maybe there should be parallelization in compression
# and a single writer point
#   thats probably the real answer...
#   aaaaaa so much work
# maybe the architecture should be different
# c samples (independent of group) -> c time indexes -> unified time axis
# then we only need one pass through the file?
#   issue would be, how to update each previous timeloc each iter
#   probably dont want to do that in memory
# temp files maybe... just could be obnoxious
#   guess it cant be huge
sum(len(k.channels) for k in md)

In [ ]:
# md[3].channels

In [ ]:
mdfc_total_size = MDFC_FIL.__sizeof__()
print(
    f'{mdfc_total_size/1000/1000:.2f} '
    'MB MDFC File'
)

In [ ]:
# ratio of mdfc vs uncompressed
cr_vs_uncomp = (MDFC_FIL.__sizeof__() / MDF_FIL.__sizeof__())
print(
     'Overall compression ratio vs uncompressed is '
    f'{cr_vs_uncomp:.3f}, or {1/cr_vs_uncomp:.2f}x'
)

In [ ]:
# ratio of mdfc vs deflate
cr_vs_deflate = (MDFC_FIL.__sizeof__() / DEFLATE_MDF_FIL.__sizeof__())
print(
     'Overall compression ratio vs Deflate-2 is '
    f'{cr_vs_deflate:.3f}, or {1/cr_vs_deflate:.2f}x'
)

In [ ]:
# does it help vs deflate-2? :) ... or :(

In [ ]:
# %%timeit
# execute decompression & compare against original
err_sig = None
err_sig_orig = None
def decompress_and_compare(sn, mdfc_fil, mdf_fil):
    global err_sig
    global err_sig_orig
    # decompress the signal from mdfc
    # and compare it against the signal in mdf
    original_sig = mdf_fil.select([sn], raw=True)[0]
    original_timestamps = original_sig.timestamps
    original_samples = original_sig.samples
    
    # decompress mdfc signal
    res = mdfc_fil.decompress_signal(sn)

    # assert all close timestamps and values
    # timestamps... may have some minor losses
    #   due to float->scaleup->int on compression
    #   i think it should be understood that the retention
    #   should be based on the scale
    try:
        assert np.allclose(
            original_timestamps,
            res.timestamps,
            # the "scaleup" applied on compression
            #   fp inaccuracies may be expected
            #   past this precision
            atol=(1/mdfc_fil.time_metadata[-1][0][1])
        ), f"{sn} timestamps not allclose!? :("
        assert np.allclose(
            original_samples,
            res.samples,
            # tolerance specification for float case
            # TODO perhaps this should be derived
            #   from compression metadata,
            #   ie the tolerance value used
            # occasionally this is not right
            atol=lossy_fp_params.get(
                'tolerance', 
                1e-20
            ),
            rtol=lossy_fp_params.get(
                'tolerance_rel',
                1e-20
            ),
            
        ), f"{sn} samples not allclose!? :("
    except:
        err_sig = res
        err_sig_orig = original_sig
        raise


if DO_TEST_DECOMPRESSION:
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass
    with (
        asammdf.MDF(MDF_FIL) as mdf_fil,
        MDFDecompressor(MDFC_FIL, 
                        close_file_on_exit=False,
                        load_time_axis_on_enter=False) as mdfc_fil
    ):
        mdfc_fil.decompress_time()
        try:
            # test signal decompression
            for sn in mdf_fil.channels_db.keys():
                if sn == 'time': continue  #
                print(f'on {sn}')
                decompress_and_compare(sn, mdfc_fil, mdf_fil)
                print(f'pass {sn}')
        except KeyError:
            print(f'{sn} found in MDF but not in compressed file...')
            raise  # ?
        else:
            print("All signals have passed decompression check :)")
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass

In [ ]:
# pass validity check :) ... or :(

In [ ]:
# lets do some time checks...
test_names = [
    # ... specific signal names...
    # 'acceleration_id'
    # 'x_value'
]
test_names = None  # all signals

In [ ]:
%%timeit
# testing the speed of reading MDF (without compression)
MDF_FIL.seek(0)
with (
    asammdf.MDF(MDF_FIL) as mfil,
):
    if test_names is None:
        sigs = [
            sig_name # (sig_name, *chan_info) 
            for sig_name, chan_info in mfil.channels_db.items()
            if sig_name != 'time'
        ]
    else:
        sigs = test_names
    for sig_sel in sigs:
        sig = mfil.select([sig_sel], raw=True)[0]

In [ ]:
%%timeit
# testing the speed of reading MDF (with deflate compression)
DEFLATE_MDF_FIL.seek(0)
with (
    asammdf.MDF(DEFLATE_MDF_FIL) as mfil,
):
    if test_names is None:
        sigs = [
            sig_name # (sig_name, *chan_info) 
            for sig_name, chan_info in mfil.channels_db.items()
            if sig_name != 'time'
        ]
    else:
        sigs = test_names
    for sig_sel in sigs:
        sig = mfil.select([sig_sel], raw=True)[0]

In [ ]:
%%timeit
# testing the speed of reading the MDFC compressed file
MDFC_FIL.seek(0)
with MDFDecompressor(MDFC_FIL, close_file_on_exit=False) as dfil:
    if test_names is None:
        sigs = dfil.channame_to_group.keys()
    else:
        sigs = test_names
    for sn in sigs:
        res = dfil.decompress_signal(sn)

In [ ]:
# end time checks :) ... or :(

In [ ]:
# thanks for playing!